# [Mixed Precision] EXAONE-4.0-1.2B Optimized Quantization

이 노트북은 **SmoothQuant (W8A8)**와 **GPTQ (W4A8)**를 혼합하여 성능(Accuracy)과 효율(Efficiency)의 균형을 맞춘 최적화 버전입니다.

**전략 (Strategy):**
- **Sensitivity Protection (BF16)**: 입출력 및 민감한 첫/마지막 레이어는 원본 정밀도 유지
- **Accuracy Layer (W8A8)**: 정확도에 민감한 **Attention (Q, K, V, O)** 레이어는 8비트 유지
- **Efficiency Layer (W4A8)**: 파라미터가 많은 **FFN (Gate, Up, Down)** 레이어는 4비트로 압축

**호환성:**
- vLLM 0.14.1+ (Native Support via `compressed-tensors`)
- `llmcompressor` >= 0.13.0

# 1. Installation

In [ ]:
# 필수 라이브러리 설치
!pip install -q compressed-tensors>=0.13.0
!pip install -q llmcompressor>=0.13.0
!pip install -q transformers>=4.57.3
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q accelerate

print("✓ Installation complete!")

# 2. Import & Setup

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# 1. 환경 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
SAVE_DIR = "EXAONE-4.0-1.2B-Mixed-0.7Plus"
DATASET_SPLIT = "train"
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 1024  # 긴 문맥 처리를 위해 1024로 상향

print(f"[INFO] Target Save Directory: {SAVE_DIR}")

# 3. Model & Tokenizer Load

In [ ]:
# 2. 모델 및 토크나이저 로드 (trust_remote_code 필수)
print("[INFO] Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("[INFO] Loading Model (BF16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print("[INFO] Model Loaded Successfully")

# 4. Data Preparation

In [ ]:
# 3. 데이터 전처리 (Chat Template 적용)
def preprocess_fn(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

print("데이터 준비 중...")
ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))
ds = ds.map(preprocess_fn)

print(f"[INFO] Calibration Dataset Ready: {len(ds)} samples")

# 5. Mixed Precision Recipe Configuration

In [ ]:
# 4. 혼합 정밀도 레시피 설계
# 0, 1, 29번 레이어 및 임베딩 층은 BF16으로 완벽 보호 (Ignore List)
IGNORE_LAYERS = [
    "embed_tokens", "lm_head",
    "model.layers.0", "model.layers.1", "model.layers.29"
]

recipe = [
    # 1. Attention 그룹: 정확도 유지를 위해 W8A8 (SmoothQuant 스타일) 적용
    GPTQModifier(
        targets=["q_proj", "k_proj", "v_proj", "o_proj"],
        scheme="W8A8",
        ignore=IGNORE_LAYERS,
        dampening_frac=0.01,
        observer="mse",
    ),
    # 2. FFN 그룹: 연산량 절감 및 속도 향상을 위해 W4A8 적용
    GPTQModifier(
        targets=["gate_proj", "up_proj", "down_proj"],
        scheme="W4A8",
        ignore=IGNORE_LAYERS,
        dampening_frac=0.01,
        observer="mse",
    )
]

print("[INFO] Mixed Precision Recipe Configured")
print(f"   - Ignored Layers: {len(IGNORE_LAYERS)} groups")
print("   - Attention: W8A8")
print("   - FFN: W4A8")

# 6. Run Quantization

In [ ]:
# 5. 양자화 실행
print(f"전략적 혼합 양자화 시작 (Samples={NUM_CALIBRATION_SAMPLES}, MaxLen={MAX_SEQUENCE_LENGTH})...")

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] Quantization Completed!")

# 7. Save & Compress

In [ ]:
# 6. 결과 저장 (vLLM 호환 형식)
print(f"모델 저장 중: {SAVE_DIR}")
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)

print("[INFO] Model Saved Successfully")

# Zip for submission
zip_name = "mixed_precision_submit"
print(f"[INFO] Creating {zip_name}.zip...")
shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=SAVE_DIR
)
print(f"[INFO] Archive Created: {zip_name}.zip")
print("모든 작업이 완료되었습니다!")